In [ ]:
import pandas as pd
import time
import pickle
import os
import random
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
#from spotipy.oauth2 import SpotifyOAuth
from spotipy.exceptions import SpotifyException

import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.config import SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET

In [ ]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean.parquet")
CACHE_PATH = Path("spotify_cache.pkl")

In [ ]:
SLEEP_TIME = 1.2
CHECKPOINT_EVERY = 100
MAX_REQUESTS = 50

In [ ]:
artist_cache = {}

In [ ]:
# LOAD DATA
df = pd.read_parquet(DATA_PATH)

unique_tracks = df[["artist_clean", "track_clean"]].drop_duplicates().reset_index(drop=True)

unique_tracks["track_id"] = (
    unique_tracks["artist_clean"] + " - " + unique_tracks["track_clean"])

In [ ]:
# LOAD CACHE
if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    print(f"Loaded cache with {len(cache)} entries")
else:
    cache = {}

In [ ]:
# SPOTIFY CLIENT
sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET
    )
)

#results = sp.search(q="artist:deftones track:change", type="track", limit=1)
#print(results)
#track = results["tracks"]["items"][0]

# print({
#     "spotify_id": track["id"],
#     "artist": track["artists"][0]["name"],
#     "track": track["name"],
#     "popularity": track["popularity"]
# })

In [ ]:
# SEARCH FUNCTION

def search_spotify(sp, artist, track):
    query = f"artist:{artist} track:{track}"
    
    while True:
        try:
            results = sp.search(q=query, type="track", limit=1)
            items = results["tracks"]["items"]
            
            if not items:
                return None
            
            t = items[0]
            artist_id = t.get("artists", [{}])[0].get("id")
            
            # 🔥 cache dla artistów
            genres = []
            if artist_id:
                if artist_id in artist_cache:
                    genres = artist_cache[artist_id]
                else:
                    try:
                        artist_data = sp.artist(artist_id)
                        genres = artist_data.get("genres", [])
                        artist_cache[artist_id] = genres
                        time.sleep(0.2)
                    except:
                        pass
            
            return {
                "spotify_id": t.get("id"),
                "spotify_artist": t.get("artists", [{}])[0].get("name"),
                "spotify_track": t.get("name"),
                "popularity": t.get("popularity"),
                "artist_id": artist_id,
                "genres": genres
            }
        
        except SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(e.headers.get("Retry-After", 5))
                time.sleep(retry_after)
            else:
                print("Spotify error:", e)
                return None
        
        except Exception as e:
            print("Connection error:", e)
            time.sleep(5)

In [ ]:
# MAIN LOOP

requests_made = 0
no_match_count = 0

for i, row in unique_tracks.iterrows():
    key = row["track_id"]
    
    if key in cache and cache[key] is not None and cache[key].get("popularity") is not None:
        continue
    
    if not row["artist_clean"] or not row["track_clean"]:
       cache[key] = None
       continue
    
    result = search_spotify(sp, row["artist_clean"], row["track_clean"])
    
    if result is None:
        no_match_count += 1
        print(f"No match: {row['artist_clean']} - {row['track_clean']}")
    
    cache[key] = result
    
    requests_made += 1
    
    if requests_made % 10 == 0:
        print(f"{requests_made} requests | i={i}")
    
    if requests_made % CHECKPOINT_EVERY == 0:
        print(f"{requests_made} requests | cache size: {len(cache)}")
        
        with open(CACHE_PATH, "wb") as f:
            pickle.dump(cache, f)
    
    if requests_made >= MAX_REQUESTS:
        print(f"Reached MAX_REQUESTS = {MAX_REQUESTS}")
        break
    
    time.sleep(1.0 + random.uniform(0.5, 1.5))

In [ ]:
list(cache.values())[0]

In [ ]:
if requests_made > 0:
    print(f"No match rate: {no_match_count / requests_made:.2%}")

In [ ]:
# FINAL SAVE

with open(CACHE_PATH, "wb") as f:
    pickle.dump(cache, f)

print("Done")

In [ ]:
clean_cache = {k: v for k, v in cache.items() if v is not None}

In [ ]:
len(cache) - len(clean_cache)

In [ ]:
spotify_df = pd.DataFrame.from_dict(clean_cache, orient="index")
spotify_df.reset_index(inplace=True)
spotify_df.rename(columns={"index": "track_id"}, inplace=True)

In [ ]:
# CREATE FEATURES DATAFRAME

ids = (
    spotify_df["spotify_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

In [ ]:
ids = ids[ids.str.len() == 22].tolist()

In [ ]:
ids = list(set(ids))

In [ ]:
print(len(ids))
print(ids[:5])

In [ ]:
spotify_df.head()

In [ ]:
spotify_df.isnull().mean()

In [ ]:
# MERGE WITH SPOTIFY DF
spotify_full = pd.concat([spotify_df.reset_index(drop=True), features_df], axis=1)

In [ ]:
df = df.merge(spotify_full, on="track_id", how="left") #TUTAJ NIE JESTEM PEWNA CZY MA BYĆ ZMERGOWANE Z DF CZY Z UNIQUE TRACKS

In [ ]:
df = df.merge(unique_tracks, on=["artist_clean", "track_clean"], how="left") #AS ABOVE